# 01 · Wrangling: read, decode and repair the raw data

This project replicates and extends Khomtchouk (2020), which predicts an organism's taxonomic group and the genome compartment of its sequences from the frequencies (not the order) of its 64 codons. It also asks whether the 21 amino-acid frequencies obtained by summing synonymous codons carry the same information with a third of the features.

The data are the UCI Codon Usage dataset (https://archive.ics.uci.edu/dataset/577/codon+usage), committed unchanged as `data/raw/codon_usage.csv`. This notebook reads the file, decodes its two label columns, finds and repairs two malformed rows, and checks the amino-acid features. Code that later notebooks reuse lives in `codon_utils.py` and is tested in `tests/`; this notebook shows why each piece is needed.

In [1]:
import pandas as pd

import codon_utils

# low_memory=False reads each column in one pass, so pandas infers one dtype
# per column instead of warning about mixed types
codon_usage = pd.read_csv('../data/raw/codon_usage.csv', low_memory=False)

In [2]:
print(codon_usage.head())
print(codon_usage.size)
print(codon_usage.shape)

  Kingdom  DNAtype  SpeciesID  Ncodons  \
0     vrl        0     100217     1995   
1     vrl        0     100220     1474   
2     vrl        0     100755     4862   
3     vrl        0     100880     1915   
4     vrl        0     100887    22831   

                               SpeciesName      UUU      UUC      UUA  \
0  Epizootic haematopoietic necrosis virus  0.01654  0.01203  0.00050   
1                         Bohle iridovirus  0.02714  0.01357  0.00068   
2             Sweet potato leaf curl virus  0.01974   0.0218  0.01357   
3             Northern cereal mosaic virus  0.01775  0.02245  0.01619   
4           Soil-borne cereal mosaic virus  0.02816  0.01371  0.00767   

       UUG      CUU  ...      CGG      AGA      AGG      GAU      GAC  \
0  0.00351  0.01203  ...  0.00451  0.01303  0.03559  0.01003  0.04612   
1  0.00678  0.00407  ...  0.00136  0.01696  0.03596  0.01221  0.04545   
2  0.01543  0.00782  ...  0.00596  0.01974  0.02489  0.03126  0.02036   
3  0.00992  0.01

This is more or less what was expected:  13028 observations of 69 variables.  With 4 distinct nucleotides and three nucleotides per codon, this gives us 4 ^ 3 = 64 variables representing the frequency of each codon.  The remaining five variables are descriptive:  kingdom, DNA type, species ID, number of codons, and species name.  

It will be important to discover what is meant by "DNA type" - check to see if there's a note or a data dictionary associated with this file.

It's also important to verify that the data types are what would be expected:  int for the species ID, DNA type, and number of codons; strings for the kingdom and species name; floating point numbers for the codon frequencies.

In [3]:
# every column that is not a float
codon_usage.dtypes[codon_usage.dtypes != 'float64']

Kingdom          str
DNAtype        int64
SpeciesID      int64
Ncodons        int64
SpeciesName      str
UUU              str
UUC              str
dtype: object

`Kingdom` and `SpeciesName` are text, and `DNAtype`, `SpeciesID` and `Ncodons` are integers, as expected. Two codon columns are not: `UUU` and `UUC` were read as text (`str`), so at least one value in each is not a number. The section *Two malformed rows* below finds them. First, the labels. The unique values of `Kingdom`:

In [4]:
codon_usage['Kingdom'].unique()

<StringArray>
['vrl', 'arc', 'bct', 'phg', 'plm', 'pln', 'inv', 'vrt', 'mam', 'rod', 'pri']
Length: 11, dtype: str

These are not kingdoms. The UCI page explains the codes: `Kingdom` is the three-letter division of the CUTG database that a sequence set comes from: 'arc' (archaea), 'bct' (bacteria), 'phg' (bacteriophage), 'plm' (plasmid), 'pln' (plant), 'inv' (invertebrate), 'vrt' (vertebrate), 'mam' (mammal), 'rod' (rodent), 'pri' (primate) and 'vrl' (virus).

Some divisions are nested: bacteriophages are viruses, primates and rodents are mammals, and mammals are vertebrates. CUTG follows the GenBank divisions, which file each entry in exactly one division, so 'vertebrate' here means "vertebrate other than mammals", 'mammal' means "mammal other than rodents and primates", and 'virus' means "virus other than bacteriophages". The GenBank plant division also holds fungi and algae. These labels are filing categories rather than a flat taxonomy, which matters when reading the kingdom confusion matrix in `05_results`.

`DNAtype` is an integer code for the genome compartment: 0 genomic, 1 mitochondrial, 2 chloroplast, 3 cyanelle, 4 plastid, 5 nucleomorph, 6 secondary endosymbiont, 7 chromoplast, 8 leucoplast, 9 "NA", 10 proplastid, 11 apicoplast, 12 kinetoplast. Code 9 needs care below: it must stay a code and not become a missing value.

Plasmids appear under `Kingdom` rather than `DNAtype`, although a plasmid is a kind of DNA molecule rather than a group of organisms.

Background on the rarer DNA types:
1.  A **cyanelle** is an organelle analogous to the chloroplast, found in glaucophytes, a grouping of freshwater algae.  
2.  **Plastids** are a type of organelle found primarily in plants.  They include **chloroplasts**, **chromoplasts**, and **leucoplasts**, which exist as other values of the DNA type variable, which is somewhat confusing - to what DNA sources, exactly, do these records refer?
3.  A **nucleomorph** is a vestigial nucleus found between the membrane pairs in some plastids.  There are only two groups of organisms in which they are known to exist.
4.  **Secondary endosymbiosis** occurs when an eukaryotic cell engulfs and absorbs another eukaryotic cell which has already developed endosymbiosis with a prokaryotic cell.  The inner prokaryotic cell is considered the **secondary endosymbiont**.
5.  **Chromoplasts** are a type of plastid used to synthesize and store carotenoid pigments, usually in flowers and fruit.
6.  **Leucoplasts** are another type of plastid, used for synthesis and storage - usually of macronutrients.  Distinguished from chloroplasts and chromoplasts by the absence of photosynthesis and pigment, as well as a more 'amoeboid' morphology.
7.  A **proplastid** is an undifferentiated plastid, formed generally in the meristem tissue of plants; all specialized plastids are derived from them.
8.  **Apicoplasts** are plastids found in most, but not all, apicomplexan parasites.  Probably a case of secondary endosymbiosis, they are believed to derive from photosynthetic plastids, and have several plant-like properties that make them good targets for drugs.  This is important, as the species carrying apicoplasts can be serious disease agents - the parasite causing malaria, for example.
9.  A **kinetoplast** is a circular complex of DNA found inside a large mitochondrion, carrying multiple copies of the mitochondrial genome.  They are known to exist only in a single phylum of flagellate protists. 

With the codes understood, look at the distribution of each label.

In [5]:
codon_usage['Kingdom'].value_counts()

Kingdom
bct    2920
vrl    2832
pln    2523
vrt    2077
inv    1345
mam     572
phg     220
rod     215
pri     180
arc     126
plm      18
Name: count, dtype: int64

It looks like we have a pretty substantial number of cases for each 'Kingdom' other than plasmids.  (Plasmids are DNA sequences found in bacteria that are not part of the bacterial chromosome and replicate independently; bacteria can exchange plasmids with each other in a process called conjugation.)  Considering that plasmid DNA is unique in comparison to the other categories for this variable, and there are a mere 18 observations, perhaps it would be appropriate to omit these cases?


In [6]:
codon_usage['DNAtype'].value_counts()

DNAtype
0     9267
1     2899
2      816
4       31
12       5
3        2
9        2
5        2
11       2
6        1
7        1
Name: count, dtype: int64

Here we see that almost all observations fall into categories 0, 1, and 2 - genomic, mitochondrial, and chloroplast DNA.  Considering that we have only a very small number of observations for the other DNA types, as well as some unanswered questions about how these categories relate to each other (some being subsets of others, etc.) - perhaps it would be reasonable to restrict our analysis to only these three categories?

Of course, another important step is to check to see how many missing values we have.

In [7]:
codon_usage.isna().values.sum()

np.int64(0)

There are no missing values as parsed, but that is not the same as every value being valid: `UUU` and `UUC` were read as text. Before finding out why, decode the two label columns with the UCI codebooks. The codebooks live in `codon_utils.py`, so every notebook uses the same mapping.

In [8]:
codon_usage['kingdom'] = codon_usage['Kingdom'].map(codon_utils.KINGDOM_CODES)
codon_usage['dna_type'] = codon_usage['DNAtype'].map(codon_utils.DNATYPE_CODES)
print('codes missing from the codebooks:', codon_usage[['kingdom', 'dna_type']].isna().sum().sum())
pd.crosstab(codon_usage['kingdom'], codon_usage['dna_type'])

codes missing from the codebooks: 0


dna_type,apicoplast,chloroplast,chromoplast,cyanelle,genomic,kinetoplast,mitochondrial,nucleomorph,plastid,secondary_endosymbiont,unknown
kingdom,,,,,,,,,,,
archaea,0,0,0,0,126,0,0,0,0,0,0
bacteria,0,0,0,0,2918,0,0,0,1,1,0
bacteriophage,0,0,0,0,220,0,0,0,0,0,0
invertebrate,2,1,0,0,922,5,411,1,3,0,0
mammal,0,0,0,0,102,0,470,0,0,0,0
plant,0,815,1,2,1523,0,152,1,27,0,2
plasmid,0,0,0,0,18,0,0,0,0,0,0
primate,0,0,0,0,83,0,97,0,0,0,0
rodent,0,0,0,0,59,0,156,0,0,0,0


The raw codes stay in the frame next to the decoded labels. `DNAtype` 9 is decoded as `'unknown'` rather than the codebook's "NA": the 2022 version wrote the string `'NA'` to a CSV, and `pd.read_csv` read it back as a missing value, so two rows (*x Doritaenopsis* sp. and *x Tritordeum* sp.) were later dropped without any record of it.

The table also shows the rows that `03_preprocessing` removes: the 18 plasmids, the 46 rows of rare DNA types (including the two code-9 rows), and one invertebrate "chloroplast" row (*Toxoplasma gondii*, an apicoplast).

## Two malformed rows

`UUU` and `UUC` were read as text. Find the values that are not numbers:

In [9]:
bad = (pd.to_numeric(codon_usage['UUU'], errors='coerce').isna()
       | pd.to_numeric(codon_usage['UUC'], errors='coerce').isna())
codon_usage.loc[bad, ['SpeciesName', 'Ncodons', 'UUU', 'UUC', 'UUA', 'UAA', 'UAG', 'UGA']]

,SpeciesName,Ncodons,UUU,UUC,UUA,UAA,UAG,UGA
486,Non-A,1238,non-B hepatitis virus,0.04362,0.0210,0.02262,0.00162,0.00000
5063,Salmonella enterica subsp. enterica serovar 4,1698,12;I,-,0.0212,0.04476,0.02473,0.00118


Both species names contain commas but are not quoted in the file: "Non-A,non-B hepatitis virus" (data row 486) and "Salmonella enterica subsp. enterica serovar 4,12;I,-" (row 5063). The CSV reader therefore split each name into two or three fields. Every line of the file still has 69 fields, so the extra name fields pushed the codon values one or two columns to the right, and the last one or two codon values are not in the file at all.

In 2022 the stray text was coerced to 0. That kept both rows in the modelling data with every codon value in the wrong column.

The missing values can be recovered exactly. Each frequency is a codon count divided by `Ncodons`, rounded to 5 decimals, so the count of the missing codon is `Ncodons` minus the counts that are present. For row 486, whose name took one extra field, the codon columns `UUC` to `UGA` hold the values of `UUU` to `UAG`, and `UGA` is missing:

In [10]:
row = codon_usage.loc[486]
present = pd.to_numeric(row[codon_utils.CODONS[1:]])  # the 63 values UUU..UAG
counts = present * row['Ncodons']
print('all counts are whole numbers:', bool((counts - counts.round()).abs().max() < 0.05))
print('codons accounted for:', int(counts.round().sum()), 'of', row['Ncodons'])
print('so UGA =', (row['Ncodons'] - int(counts.round().sum())), '/', row['Ncodons'],
      '=', round((row['Ncodons'] - counts.round().sum()) / row['Ncodons'], 5))

all counts are whole numbers: True
codons accounted for: 1235 of 1238
so UGA = 3 / 1238 = 0.00242


Why the *last* codons are missing and not the first: if the values were aligned to the right instead, row 486 would use the stop codon UAA for 2.3% of all codons, and row 5063 would use UAA for 4.5% and UAG for 2.5%. Stop codons end genes, so they are typically a few tenths of a percent of codons. Aligned to the left, the two rows have 0.40% and 0.12% stop codons.

`codon_utils.read_raw` applies this repair (`repair_row`) to any row whose codon fields are not all numeric, and refuses a row it cannot repair exactly. It also adds the decoded `kingdom` and `dna_type` columns and a `repaired` flag:

In [11]:
raw = codon_utils.read_raw('../data/raw/codon_usage.csv')
raw.loc[raw['repaired'], ['SpeciesName', 'Ncodons', 'UUU', 'UUC', 'UAA', 'UAG', 'UGA']]

,SpeciesName,Ncodons,UUU,UUC,UAA,UAG,UGA
486,"Non-A,non-B hepatitis virus",1238,0.04362,0.02100,0.00162,0.0,0.00242
5063,"Salmonella enterica subsp. enterica serovar 4,...",1698,0.02120,0.02356,0.00118,0.0,0.00000


In [12]:
sums = raw[codon_utils.CODONS].sum(axis=1)
print(f'codon frequencies sum to between {sums.min():.5f} and {sums.max():.5f} in every row')
print(f'row 486 sums to {sums[486]:.5f}')

codon frequencies sum to between 0.99984 and 1.00017 in every row
row 486 sums to 1.00000


After the repair every row sums to 1 within rounding. With the 2022 zero-fill, row 486 summed to 0.99758, the outlier that a row-sum check would have caught.

## Amino-acid features

Each amino acid is encoded by one to six synonymous codons, and three codons are stop signals. Summing codon frequencies within each group gives 21 features: 20 amino acids and *stop*. The 2022 version also had a *start* column, but the start codon AUG is the methionine codon, so *start* was an exact copy of *methionine*. It is gone. Because the 21 groups partition the 64 codons, each row's amino-acid frequencies sum to the same total as its codon frequencies.

In [13]:
print(', '.join(f'{amino} ({len(codons)})' for amino, codons in codon_utils.AMINO_CODONS.items()))
amino = codon_utils.amino_features(raw)
amino.head()

alanine (4), arginine (6), asparagine (2), aspartic_acid (2), cysteine (2), glutamine (2), glutamic_acid (2), glycine (4), histidine (2), isoleucine (3), leucine (6), lysine (2), methionine (1), phenylalanine (2), proline (4), serine (6), threonine (4), tryptophan (1), tyrosine (2), valine (4), stop (3)


,alanine,arginine,asparagine,aspartic_acid,cysteine,glutamine,glutamic_acid,glycine,histidine,isoleucine,...,lysine,methionine,phenylalanine,proline,serine,threonine,tryptophan,tyrosine,valine,stop
0,0.08673,0.06817,0.03008,0.05615,0.01404,0.03409,0.05564,0.08722,0.02857,0.03308,...,0.04913,0.02506,0.02857,0.07268,0.06115,0.04561,0.01003,0.02757,0.09423,0.00301
1,0.08548,0.06378,0.03799,0.05766,0.01899,0.02781,0.05970,0.06716,0.02171,0.04545,...,0.03935,0.03324,0.04071,0.05020,0.06581,0.05768,0.01425,0.03392,0.08955,0.00339
2,0.05594,0.07445,0.04751,0.05162,0.02859,0.03887,0.04710,0.05779,0.03291,0.05203,...,0.06026,0.02180,0.04154,0.06232,0.08289,0.05491,0.01728,0.03887,0.05780,0.00535
3,0.05587,0.04596,0.03656,0.05692,0.01880,0.02820,0.06475,0.06423,0.01462,0.07154,...,0.07415,0.02924,0.04020,0.03760,0.09191,0.06215,0.01201,0.03029,0.07885,0.00418
4,0.06601,0.05789,0.04534,0.06631,0.01787,0.03311,0.07038,0.05658,0.02032,0.04893,...,0.07372,0.02773,0.04187,0.02798,0.06893,0.06745,0.01205,0.03456,0.07196,0.00175


In [14]:
print('columns:', amino.shape[1])
print('largest |amino sum - codon sum|:', (amino.sum(axis=1) - sums).abs().max())
print('methionine is AUG:', bool((amino['methionine'] == raw['AUG']).all()))

columns: 21
largest |amino sum - codon sum|: 1.1102230246251565e-15
methionine is AUG: True


Nothing is written to disk. Later notebooks call `codon_utils.read_raw` themselves, so there is one source of truth and no derived CSV to keep in sync (the 2022 `codon_usage2.csv` was a 10 MB derived copy of this file).